# Ext_Debt_Data via `lic_dsf.pv`

Goal: recreate **Ext_Debt_Data** new-MLT functionality by initializing every template debt instrument from `lic-dsf-template-2025-08-12.xlsx`, then aggregating through a portfolio.

# Ext_Debt_Data via `lic_dsf.pv`

Goal: recreate **Ext_Debt_Data** new-MLT functionality by initializing every template debt instrument from `lic-dsf-template-2025-08-12.xlsx`, then aggregating through a portfolio.

| Layer | Class / API | Role | Status |
|-------|-------------|------|--------|
| Instrument | `PresentValueInstrument` | One new loan (`internal` / `external`) | works |
| Load | `load_instruments_from_workbook` | Input 4 terms + disbursements → instruments | works |
| Portfolio | `PVPortfolio` | Owns instruments → new-debt aggregates | works |
| Book | `ExternalDebtBook` | Old debt + ST/SDR + Ext_Debt headlines | later |

See `docs/ext-debt-module-design.md`.

See `docs/ext-debt-module-design.md`.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent

WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"

pd.set_option("display.max_columns", 24)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_rows", 80)

WORKBOOK

PosixPath('/home/sravan/excel-grapher/LICDSF-extraction-pipeline/data/lic-dsf-template-2025-08-12.xlsx')

## 1. Load every Input 4 / PV_Base instrument from the template

`include_zero_disbursement=True` so the Ext_Debt creditor list is complete (many lines are zero in this template).

```python
instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=True,
)
# each item is a PresentValueInstrument
```

In [2]:
from lic_dsf.pv import load_instruments_from_workbook

instruments = load_instruments_from_workbook(
    WORKBOOK,
    include_zero_disbursement=True,
)

catalog = pd.DataFrame(
    [
        {
            "name": i.name,
            "grace": i.grace,
            "maturity": i.maturity,
            "interest": i.interest_rate,
            "discount": i.discount_rate,
            "disbursement_sum": sum(i.disbursements),
            "n_years": len(i.disbursements),
        }
        for i in instruments
    ]
)
print(f"{len(instruments)} instruments from {WORKBOOK.name}")
catalog

30 instruments from lic-dsf-template-2025-08-12.xlsx


,name,grace,maturity,interest,discount,disbursement_sum,n_years
0,IMF,5,10,0.0025,0.0500,0.0000,21
1,IDA - regular,6,38,0.0075,0.0500,790.3180,21
2,IDA - 50Y loans,10,50,0.0000,0.0500,300.0000,21
3,IDA - SML,6,12,0.0000,0.0500,10.0000,21
4,IDA NEW 40-year credits,11,40,0.0000,0.0500,10.0000,21
5,IDA NEW Regular,6,31,0.0075,0.0500,100.0000,21
6,IDA NEW Blend (also enter) -->,5,25,0.0324,0.0500,120.0000,21
7,IDA NEW 60-year credits,20,60,0.0000,0.0500,120.0000,21
8,MULTI1,5,30,0.0075,0.0500,0.0000,21
9,MULTI2,5,30,0.0000,0.0500,0.0000,21


## 2. Spot-check one instrument Output (works today once loaded)

Pick a nonzero line (Eurobond in this template) and show the Ext_Debt-facing Output rows: Interest, Amortization, PV, Stock.

In [3]:
by_name = {i.name: i for i in instruments}
sample_name = "Eurobond" if "Eurobond" in by_name else instruments[0].name
sample = by_name[sample_name]

external = sample.external()
ext_debt_rows = external.loc[
    [
        "Interest",
        "Amortization",
        f"PV of debt   {sample.name}",
        "Stock of new forex debt (in USD)",
    ]
]
print(sample_name, "→ Ext_Debt Output metrics")
ext_debt_rows.iloc[:, :10]

Eurobond → Ext_Debt Output metrics


,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
Interest,0.0000,0.0000,0.0000,0.0000,22.5000,45.0000,67.5000,157.5000,187.5000,217.5000
Amortization,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
PV of debt Eurobond,0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3333","2,416.6667","2,750.0000"
Stock of new forex debt (in USD),0.0000,0.0000,0.0000,250.0000,500.0000,750.0000,"1,750.0000","2,083.3333","2,416.6667","2,750.0000"


## 3. `PVPortfolio` — own instruments, build new-debt aggregates

```python
portfolio = PVPortfolio(instruments)
portfolio.aggregate_external()
portfolio.interest()
portfolio.amortization()
portfolio.pv()
portfolio.stock()
portfolio.new_debt_service()
```

These are the Ext_Debt **new MLT** panels (Interest / Amortization / PV / Stock totals), before old debt or DSA headlines.

In [4]:
from lic_dsf.pv import PVPortfolio

portfolio = PVPortfolio(tuple(instruments))

totals = portfolio.aggregate_external()
assert isinstance(totals, pd.DataFrame)

display(totals.iloc[:, :10])

interest = portfolio.interest()
amortization = portfolio.amortization()
pv = portfolio.pv()
stock = portfolio.stock()
new_ds = portfolio.new_debt_service()

print("interest shape:", interest.shape)
print("amortization shape:", amortization.shape)
print("pv shape:", pv.shape)
print("stock shape:", stock.shape)
new_ds.iloc[:, :10]

,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
"New forex borrowing (gross, USD)","1,071.9440",984.3925,"1,210.6787","1,594.3500","1,673.9300","1,787.4800","2,352.8957","1,962.6378","2,241.8368","2,325.6747"
cumulative,"1,071.9440","2,056.3364","3,267.0151","4,861.3651","6,535.2951","8,322.7751","10,675.6709","12,638.3086","14,880.1455","17,205.8202"
Stock of new forex debt (in USD),"1,071.9440","2,056.3364","3,179.5151","4,686.3651","6,245.4861","7,884.9789","10,141.8928","11,965.5436","13,971.4098","15,896.4733"
PV of debt,899.1668,"1,620.9792","2,360.3225","3,495.0002","4,822.3646","6,319.3737","8,559.8699","10,313.0252","12,260.7004","14,139.9341"
Total debt service (in USD),0.0000,37.5339,150.0686,171.3496,248.3110,342.8919,363.9449,550.2004,735.8611,999.0116
Interest,0.0000,37.5339,62.5686,83.8496,133.5020,194.9047,267.9632,411.2133,499.8905,598.4003
Amortization,0.0000,0.0000,87.5000,87.5000,114.8090,147.9873,95.9818,138.9870,235.9706,400.6112


interest shape: (30, 61)
amortization shape: (30, 61)
pv shape: (30, 61)
stock shape: (30, 61)


,2024,2025,2026,2027,2028,2029,2030,2031,2032,2033
Interest,0.0000,37.5339,62.5686,83.8496,133.5020,194.9047,267.9632,411.2133,499.8905,598.4003
Amortization,0.0000,0.0000,87.5000,87.5000,114.8090,147.9873,95.9818,138.9870,235.9706,400.6112
Total new debt service,0.0000,37.5339,150.0686,171.3496,248.3110,342.8919,363.9449,550.2004,735.8611,999.0116


## 4. `ExternalDebtBook` — full Ext_Debt_Data (later)

Wires `PVPortfolio` new-MLT aggregates to old MLT NPV, arrears, ST, SDR, and headline totals (`total_pv_of_debt`, public debt service, PPG check).

```python
book = ExternalDebtBook(portfolio=portfolio, inputs=load_external_debt_inputs(WORKBOOK))
book.summary()
```

In [6]:
from lic_dsf.pv import ExternalDebtBook, load_external_debt_inputs

book = ExternalDebtBook(
    portfolio=portfolio,
    inputs=load_external_debt_inputs(WORKBOOK),
)
book.summary().iloc[:, :10]

ImportError: cannot import name 'ExternalDebtBook' from 'lic_dsf.pv' (/home/sravan/excel-grapher/LICDSF-extraction-pipeline/src/lic_dsf.pv/__init__.py)

## Build order

1. `load_instruments_from_workbook` — all Input 4 / PV_Base lines as `PresentValueInstrument`
2. `PVPortfolio` — own them; `interest` / `amortization` / `pv` / `stock` / `new_debt_service`
3. Parity vs `Ext_Debt_Data` new-MLT rows (F142 / F192 / F279 / F329)
4. `ExternalDebtBook` + old debt / headlines
5. LC-NR / resident FX extras if needed for full sheet parity